# Step 3 — MLP on raw ISI-distribution histograms (AD vs WT)

| | |
|---|---|
| **Original** | `Liam_work/tests/behavior_isi_analysis/Raw_ISI_MLP_Classification.ipynb` |
| **Status** | **Main analysis #2** (last edited 2026-09-25). Same experiment grid and validation as step 2, different input representation and model. |

**Representation.** For each neuron and state, keep ISIs in (0, 60 s], histogram them into **100
log-spaced bins from 1 ms to 60 s**, and min–max normalise each histogram to [0, 1] (so only the *shape*
inside the bin range is kept — not spike count, rate scale, or ISI order). Three states → 300 features.
Neurons with < 10 ISIs in a state get an all-zero histogram for that state.

**Model.** scikit-learn `MLPClassifier` (ReLU, Adam, early stopping). Per fold: NaN→train median,
standardise on training rows, randomly oversample the minority class. Hyperopt (30 evals, 3-fold
StratifiedGroupKFold) chooses architecture, L2 `alpha`, learning rate and batch size.

**Label / cohorts / exclusions / L2O / metrics** — identical to the XGBoost notebook: genotype label,
ALL / OLD (> 200 d) / YOUNG cohorts, `caf104` dropped, `caf42` dropped from tuning + L2O,
one WT + one AD animal held out per fold, `N_Acc` / `N_BAcc` / `A_Acc`.

**Sections.**
1. Setup + load raw ISI table (13 GB)
2. `fit_mlp` helper
3. Build 300-bin histogram features
4. Hyperopt search → `BEST_MLP_PARAMS`
5. L2O over every (age split × state / region / cell type); held-out permutation importance for the
   ALL-animal sleep-state models
6. Per-bin permutation importance figures (+ saves `mlp_bin_importance.csv`)
7. Result figures 1–4 and summary table
8. Fig 3B (score vs age, by sex)
9. QC deep-dive on `caf73` (a hard-to-classify old AD animal)
10. Restricted-bin MLPs: does the most important ISI window alone carry the signal? (vs equal-sized
    least-important and random-bin controls)
11. Exploratory PCA / LDA / k-means-ARI of the histograms (after Tolossa et al., eLife 2025)

**Figures/CSV written next to this notebook:** `fig_mlp_*.png`, `fig1_forest`, `fig2_boxplot_folds`,
`fig3_age_confidence[_sex]`, `fig4*_*`, `investigate_caf73_histogram`, `mlp_bin_importance.csv`.

**Runtime.** ~1 min to load, ~40 s histograms, ~2–3 min tuning, ~3 min L2O.

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

# AD vs WT Classification: Raw ISI MLP
## Direct ISI Histogram Classification — No Hand-Crafted Features

### Strategy
Feed raw ISI distributions (as log-spaced histogram bins) directly to the MLP.
No hand-crafted statistical features — let the network learn what matters from the raw shape of the ISI distribution.

### Representation
- Each neuron's ISI values (per sleep/wake state) are binned into **100 log-spaced bins** (0.001s to 60s)
- Min-Max normalized histograms [0,1] → **300 features** (100 bins × 3 states: Wake, NREM, REM)
- This represents distributional shape within the bin range, discarding interval order and total counts

### Tests
- **All animals** (L2O CV)
- **OLD only** (>200 days): WT vs AD with known phenotype
- **YOUNG only** (≤200 days): WT vs AD pre-symptomatic detection

### Dataset
- `all_animal_isi_df.pkl`: ~5,500 neurons from 20 animals (11 WT, 9 AD)
- **Excluded:** `caf104` — 81% of neurons have within-epoch ISIs >100s (massive outlier)
- Age cutoff: 200 days

Validation holds out one WT and one AD animal per fold (L2O). Hyperparameters come
from one hyperopt search on all analysed animals, which include the L2O test
animals: report as "hyperparameters selected using all animals". Scores are uncalibrated model
outputs. Age trends and importance-rank correlations are descriptive.
`caf42` remains excluded at user request; its biological rationale is pending.
Sex coloring does not adjust for sex, and these data do not establish progression.
Neuron rows can span multiple recording ages; animal IDs define every split.
The two representations retain their different ISI support (MLP: 1 ms–60 s;
metric tables: all positive finite ISIs), so this is not a model-only comparison.


## 1. Setup and data loading
Preamble, figure style, knobs (`AGE_CUTOFF = 200`, `N_JOBS = 20`, `EXCLUDE_ANIMALS = ['caf104']`, `SUBSET_EXCLUDE = ['caf42']`), then the 13 GB raw ISI table.

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (Set BEFORE importing numpy/scipy/sklearn/xgboost so BLAS picks them up)
# =============================================================================
import os
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'
os.environ['PYTHONWARNINGS']         = 'ignore'   # also suppress in joblib workers

import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# =============================================================================
# SETUP AND DATA LOADING
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             confusion_matrix)
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE
from joblib import Parallel, delayed, parallel_backend
import time
import warnings
warnings.filterwarnings('ignore')


# =============================================================================
# FIGURE STYLE — matches the lab's prey-capture paper (Schneider, McGregor et al.):
# Arial, open axes, no gridlines, light-grey boxes with colored per-animal points,
# black-edged legends and statistic boxes, panel letters, captions printed below.
# =============================================================================
import matplotlib.lines as mlines
from matplotlib.patches import Patch

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False,
    'axes.linewidth': 0.8, 'axes.edgecolor': 'black', 'axes.titlepad': 6,
    'xtick.major.size': 3, 'ytick.major.size': 3, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.minor.size': 1.5, 'ytick.minor.size': 1.5, 'xtick.minor.width': 0.6, 'ytick.minor.width': 0.6,
    'lines.linewidth': 1.5, 'patch.linewidth': 0.8,
    'legend.frameon': True, 'legend.fancybox': True, 'legend.edgecolor': 'black',
    'legend.framealpha': 1.0, 'legend.handlelength': 1.2, 'legend.handletextpad': 0.5,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight', 'savefig.facecolor': 'white',
})

# Colour-blind-checked palettes (CVD separation >= 12 ΔE for every pair)
GENO_COLORS  = {'WT': '#5B84B1', 'AD': '#D9826B'}                  # steel blue / terracotta
SEX_COLORS   = {('WT', 'M'): '#264F7D', ('WT', 'F'): '#8DB6E0',
                ('AD', 'M'): '#9E4631', ('AD', 'F'): '#F2A88C'}
STATE_COLORS = {'wake': '#E8B33A', 'nrem': '#7E5BA6', 'rem': '#2F8F5B'}
STATE_NAMES  = {'wake': 'Wake', 'nrem': 'NREM', 'rem': 'REM'}
GROUP_COLORS = {'Sleep State': '#3C3C3C', 'Brain Region': '#2F9BB5', 'Cell Type': '#BDBDBD'}
BOX_FILL, BOX_EDGE, REF_GREY = '#EBEBEB', '#6E6E6E', '#9E9E9E'


def panel_letters(fig, axes, letters='ABCDEFGHIJKL'):
    """Paper-style panel letters at the top-left of each panel (call once layout is final)."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    to_fig = fig.transFigure.inverted()
    for letter, ax in zip(letters, np.ravel(axes)):
        box = to_fig.transform_bbox(ax.get_tightbbox(renderer))
        fig.text(box.x0, box.y1 + 0.01, letter, fontsize=12, ha='left', va='bottom')


def stat_box(ax, text, x=0.97, y=0.03, ha='right', va='bottom'):
    """Square black-edged annotation box, as used for statistics in the paper."""
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontsize=7.5, linespacing=1.3,
            bbox=dict(boxstyle='square,pad=0.4', facecolor='white', edgecolor='black', linewidth=0.8))


def genotype_handles(marker='o'):
    """Legend entries for the two genotypes."""
    return [mlines.Line2D([], [], color=GENO_COLORS[g], marker=marker, ls='', ms=5, label=lab)
            for g, lab in [('AD', 'AD (APP/PS1)'), ('WT', 'WT')]]


CHANCE_HANDLE = mlines.Line2D([], [], color='black', ls='--', lw=0.8, label='Chance (50%)')
np.random.seed(42)

AGE_CUTOFF = 200  # days
N_JOBS = 20       # use 20 of 104 cores
SUBSET_EXCLUDE = ['caf42']  # retained at user request; biological rationale pending
EXCLUDE_ANIMALS = ['caf104']  # documented long-ISI QC concern

# Load data
print('Loading dataset (this takes a moment - 13 GB file)...')
t0 = time.time()
df = pd.read_pickle('/media/HlabShare/Ben_work/isi/all_animal_isi_df.pkl')

# Exclude outlier animals
n_before = len(df)
df = df[~df['animal'].isin(EXCLUDE_ANIMALS)].reset_index(drop=True)
print('Excluded {}: dropped {:,} neurons ({:,} -> {:,})'.format(
    EXCLUDE_ANIMALS, n_before - len(df), n_before, len(df)))

df['age_group'] = df['age'].apply(lambda x: 'OLD' if x > AGE_CUTOFF else 'YOUNG')
df['geno_label'] = df['geno'].map({'wt': 'WT', 'app_ps1': 'AD'})
df['group'] = df['geno_label'] + '_' + df['age_group']
# CLASSIFICATION TARGET = genotype only (APP/PS1 -> 1, WT -> 0).
# Age is NOT in the label; 'age_group' is only used to split cohorts (ALL / OLD / YOUNG).
df['label'] = (df['geno'] == 'app_ps1').astype(int)

print('Dataset loaded in {:.1f}s.'.format(time.time() - t0))
print('Total neurons: {:,}'.format(len(df)))
print('Animals: {}'.format(df['animal'].nunique()))
print('Age cutoff: {} days | Using {} worker(s)'.format(AGE_CUTOFF, N_JOBS))
print()
for grp in ['WT_YOUNG', 'AD_YOUNG', 'WT_OLD', 'AD_OLD']:
    sub = df[df['group'] == grp]
    print('  {}: {} animals, {:,} neurons, ages {:.0f}-{:.0f}d'.format(
        grp, sub['animal'].nunique(), len(sub),
        sub['age'].min(), sub['age'].max()))

## 2. Shared helpers
`clean_isi_array` concatenates a neuron's per-epoch ISI arrays keeping 0 < ISI ≤ 60 s. `fit_mlp` is the one place a model is trained (median-impute → scale → oversample → MLP), used by tuning and by every L2O fold.

In [ ]:
# =============================================================================
# SHARED UTILITIES
# =============================================================================

def clean_isi_array(isi_list, min_isi=0, max_isi=60):
    """Concatenate ISI values from the epoch list, keeping min_isi < ISI <= max_isi."""
    kept = [arr[(arr > min_isi) & (arr <= max_isi)]
            for arr in isi_list if isinstance(arr, np.ndarray) and len(arr) > 0]
    return np.concatenate(kept) if kept else np.array([])


def fit_mlp(params, X_train, y_train, max_iter=300, seed=42):
    """Train one MLP on training rows only. Returns (model, transform for new rows).

    NaNs are filled with training medians and features are scaled on the training
    rows; the minority class is then randomly oversampled to match the majority.
    `params` (hidden_layer_sizes, alpha, learning_rate_init, batch_size) is passed in
    explicitly because parallel workers cannot see notebook globals.
    """
    medians = np.nan_to_num(np.nanmedian(X_train, axis=0))
    scaler = StandardScaler().fit(np.where(np.isnan(X_train), medians, X_train))

    def transform(X):
        return scaler.transform(np.where(np.isnan(X), medians, X))

    X_fit, y_fit = transform(X_train), y_train
    counts = np.bincount(y_fit, minlength=2)
    minority = int(np.argmin(counts))
    if 0 < counts[minority] < counts.max():
        rng = np.random.RandomState(seed)
        extra = rng.choice(np.where(y_fit == minority)[0],
                           size=counts.max() - counts[minority], replace=True)
        X_fit = np.vstack([X_fit, X_fit[extra]])
        y_fit = np.concatenate([y_fit, y_fit[extra]])

    mlp = MLPClassifier(**params, activation='relu', solver='adam',
                        learning_rate='adaptive', max_iter=max_iter,
                        early_stopping=True, validation_fraction=0.15,
                        n_iter_no_change=10, random_state=seed)
    mlp.fit(X_fit, y_fit)
    return mlp, transform


## 3. ISI histogram features
100 log-spaced bins (1 ms – 60 s) per state, min–max normalised per neuron per state → `raw_isi_df` with columns `{wake,nrem,rem}_bin{0..99}`.

In [ ]:
# =============================================================================
# RAW ISI HISTOGRAM REPRESENTATION (PARALLELIZED)
# =============================================================================
print('=' * 70)
print('BUILDING RAW ISI HISTOGRAM REPRESENTATION')
print('=' * 70)

N_BINS = 100
BIN_EDGES = np.logspace(np.log10(0.001), np.log10(60), N_BINS + 1)


def isi_to_histogram(isi_list, bin_edges=BIN_EDGES):
    """Convert raw ISI arrays to a Min-Max normalized histogram [0, 1]."""
    isis = clean_isi_array(isi_list)
    if len(isis) < 10:
        return np.zeros(len(bin_edges) - 1)
    hist, _ = np.histogram(isis, bins=bin_edges)
    hist = hist.astype(float)
    h_min = hist.min()
    h_max = hist.max()
    if h_max > h_min:
        hist = (hist - h_min) / (h_max - h_min)
    elif h_max > 0:
        # All bins equal and nonzero — flat distribution
        hist = np.ones_like(hist)
    return hist


def _process_raw_isi_row(row):
    """Process one row for raw ISI — for parallel execution."""
    record = {
        'animal': row['animal'],
        'geno_label': 'AD' if row['geno'] == 'app_ps1' else 'WT',
        'age': row['age'], 'age_group': row['age_group'],
        'brain_region': row['brain_region'], 'cell_type': row['cell_type'],
        'peak_latency': row['peak_latency'],
        'label': 1 if row['geno'] == 'app_ps1' else 0
    }
    for state in ['wake', 'nrem', 'rem']:
        hist = isi_to_histogram(row['{}_isi'.format(state)])
        for b in range(len(hist)):
            record['{}_bin{}'.format(state, b)] = hist[b]
    return record


print('Converting ISI arrays to {}-bin log-spaced histograms...'.format(N_BINS))
print('Bin range: {:.1f}ms to {:.1f}s'.format(BIN_EDGES[0] * 1000, BIN_EDGES[-1]))
print('Normalization: Min-Max [0, 1] per neuron per state')

t0 = time.time()
rows_list = [row for _, row in df.iterrows()]
raw_isi_records = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(_process_raw_isi_row)(row) for row in rows_list
)

raw_isi_df = pd.DataFrame(raw_isi_records)
raw_isi_feat_cols = [c for c in raw_isi_df.columns if '_bin' in c]
print('Histogram extraction: {:.1f}s'.format(time.time() - t0))
print('Raw ISI features: {} ({} bins x 3 states)'.format(len(raw_isi_feat_cols), N_BINS))

## 4. Hyperparameter search (hyperopt / TPE)
Searches architecture, `alpha`, learning rate and batch size on 3 grouped folds; also scores the old defaults for comparison. Result: `BEST_MLP_PARAMS`.

In [ ]:
# =============================================================================
# MLP — Hyperopt Hyperparameter Tuning  (lab convention)
# =============================================================================
# Uses hyperopt (TPE search) — the lab's standard tuning library.
# Inner CV: 3-fold StratifiedGroupKFold (stratify by genotype, group by animal),
# with the 3 folds of each trial trained in parallel. Tuned on all analysed
# animals (caf42 excluded), so the L2O test animals below are part of this
# search: report as "hyperparameters selected using all animals".
# BEST_MLP_PARAMS is passed explicitly to every L2O fold.

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score

# --- knobs -------------------------------------------------------------------
MAX_EVALS          = 30       # MLPs are slow — keep this lean
HYPEROPT_CV_SPLITS = 3
RANDOM_SEED        = 42

# --- prep data ---------------------------------------------------------------
df_opt = raw_isi_df[~raw_isi_df['animal'].isin(SUBSET_EXCLUDE)].reset_index(drop=True)
X_opt = df_opt[raw_isi_feat_cols].values.astype(float)
y_opt = df_opt['label'].values
groups_opt = df_opt['animal'].values
cv_splits = list(StratifiedGroupKFold(n_splits=HYPEROPT_CV_SPLITS, shuffle=True,
                                      random_state=RANDOM_SEED).split(X_opt, y_opt, groups_opt))

print(f'Hyperopt setup: {len(df_opt)} neurons, {df_opt["animal"].nunique()} animals, '
      f'{len(raw_isi_feat_cols)} features, {MAX_EVALS} evals.')
print(f'Inner CV: {HYPEROPT_CV_SPLITS}-fold StratifiedGroupKFold.')

# Architecture choices (indexed by hp.choice)
ARCH_CHOICES = [
    (128, 64), (256, 128), (256, 128, 64),
    (128, 64, 32), (384, 192, 96),
]
BATCH_CHOICES = [128, 256, 512]

space = {
    'arch_idx':           hp.choice('arch_idx', list(range(len(ARCH_CHOICES)))),
    'alpha':              hp.loguniform('alpha', np.log(1e-5), np.log(1e-1)),
    'learning_rate_init': hp.loguniform('learning_rate_init', np.log(1e-4), np.log(1e-2)),
    'batch_idx':          hp.choice('batch_idx', list(range(len(BATCH_CHOICES)))),
}


def _to_mlp_params(p):
    """Translate a hyperopt sample into MLPClassifier keyword arguments."""
    return {
        'hidden_layer_sizes': ARCH_CHOICES[int(p['arch_idx'])],
        'alpha':              float(p['alpha']),
        'learning_rate_init': float(p['learning_rate_init']),
        'batch_size':         BATCH_CHOICES[int(p['batch_idx'])],
    }


def _cv_fold_score(params, X_tr, y_tr, X_va, y_va):
    """Balanced accuracy of one inner-CV fold (candidates train for at most 120 epochs)."""
    mlp, transform = fit_mlp(params, X_tr, y_tr, max_iter=120)
    return balanced_accuracy_score(y_va, mlp.predict(transform(X_va)))


def mlp_objective(p):
    params = _to_mlp_params(p)
    scores = Parallel(n_jobs=HYPEROPT_CV_SPLITS)(
        delayed(_cv_fold_score)(params, X_opt[tr], y_opt[tr], X_opt[va], y_opt[va])
        for tr, va in cv_splits)
    # hyperopt minimizes — return negative score
    return {'loss': -float(np.mean(scores)), 'status': STATUS_OK}


# Untuned reference: the defaults this notebook used before tuning, scored on the same CV folds.
DEFAULT_MLP_PARAMS = {'hidden_layer_sizes': (128, 64), 'alpha': 0.01,
                      'learning_rate_init': 0.001, 'batch_size': 256}
with parallel_backend('loky', inner_max_num_threads=1):
    default_scores = Parallel(n_jobs=HYPEROPT_CV_SPLITS)(
        delayed(_cv_fold_score)(DEFAULT_MLP_PARAMS, X_opt[tr], y_opt[tr], X_opt[va], y_opt[va])
        for tr, va in cv_splits)
default_score = float(np.mean(default_scores))
print(f'Default settings ({DEFAULT_MLP_PARAMS["hidden_layer_sizes"]}, alpha={DEFAULT_MLP_PARAMS["alpha"]}): '
      f'balanced_acc = {default_score:.4f}')

print(f'Running {MAX_EVALS} hyperopt trials...')
t0 = time.time()
trials = Trials()
with parallel_backend('loky', inner_max_num_threads=1):   # 1 BLAS thread per worker
    best_raw = fmin(
        fn        = mlp_objective,
        space     = space,
        algo      = tpe.suggest,
        max_evals = MAX_EVALS,
        trials    = trials,
        rstate    = np.random.default_rng(RANDOM_SEED),
    )
BEST_MLP_PARAMS = _to_mlp_params(space_eval(space, best_raw))
best_score = -min(t['result']['loss'] for t in trials.trials)

print(f'\nSearch finished in {time.time() - t0:.0f}s')
print(f'Best trial: balanced_acc = {best_score:.4f}')
print('Best hyperparameters:')
for k, v in BEST_MLP_PARAMS.items():
    print(f'  {k:<24} = {v:.5g}' if isinstance(v, float) else f'  {k:<24} = {v}')

# Convergence plot
neg_scores = [-t['result']['loss'] for t in trials.trials]
best_so_far = np.maximum.accumulate(neg_scores)

trial_no = np.arange(1, len(neg_scores) + 1)
fig, ax = plt.subplots(figsize=(3.4, 2.4))
ax.scatter(trial_no, neg_scores, s=12, color=REF_GREY, linewidths=0, label='Trial')
ax.plot(trial_no, best_so_far, color='black', lw=1.5, label='Running best')
ax.axhline(default_score, color=REF_GREY, ls='--', lw=1.0, label='Default settings')
ax.set_xlabel('Hyperopt trial')
ax.set_ylabel(f'Balanced accuracy\n({HYPEROPT_CV_SPLITS}-fold grouped CV)')
ax.set_title(f'MLP hyperparameter search\n'
             f'best {best_score:.3f} vs default {default_score:.3f} '
             f'({best_score - default_score:+.3f})', fontsize=8)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.savefig('fig_mlp_hyperopt.png')
plt.show()
print(f'Tuning gained {best_score - default_score:+.4f} balanced accuracy over the defaults.')
print('Saved: fig_mlp_hyperopt.png')


## 5. Leave-two-animals-out runs for every (age split × config)
State configs restrict the *columns* (that state's 100 bins); region / cell-type configs restrict the *rows* and use all 300 bins. For the four ALL-animal state models, each fold also computes held-out permutation importance (drop in balanced accuracy when one bin is shuffled).

In [ ]:
# =============================================================================
# COMPREHENSIVE FEATURE SUBSET ANALYSIS — SETUP & RUN  (LEAVE-TWO-ANIMAL-OUT)
# =============================================================================
# Each fold holds out ONE WT + ONE AD animal. With 9 WT and 9 AD we get
# 9*9 = 81 folds for the ALL split. Each animal appears in multiple folds.
# Every fold uses BEST_MLP_PARAMS from the hyperopt cell, passed explicitly.

N_REPEATS = 10   # permutation-importance repeats (ALL-animal sleep-state models only)

# Discover cell types present in the dataset
CELL_TYPES = sorted(raw_isi_df['cell_type'].dropna().unique().tolist())
print('Cell types discovered:', CELL_TYPES)
print('Brain regions:', sorted(raw_isi_df['brain_region'].dropna().unique().tolist()))
print(f'Animals excluded from subset analysis: {SUBSET_EXCLUDE}')
print()

# Feature subsets: vary which sleep-state bins are included as features
state_feat_map = {
    'All States': raw_isi_feat_cols,
    'Wake':       [c for c in raw_isi_feat_cols if c.startswith('wake_')],
    'NREM':       [c for c in raw_isi_feat_cols if c.startswith('nrem_')],
    'REM':        [c for c in raw_isi_feat_cols if c.startswith('rem_')],
}


def _l2o_pairs(groups, y):
    """Yield (train_idx, test_idx, test_animals) for every (WT, AD) pair."""
    uniq = pd.DataFrame({'animal': groups, 'label': y}).drop_duplicates()
    wt_animals = sorted(uniq[uniq['label'] == 0]['animal'].unique().tolist())
    ad_animals = sorted(uniq[uniq['label'] == 1]['animal'].unique().tolist())
    for wt in wt_animals:
        for ad in ad_animals:
            test_mask = (groups == wt) | (groups == ad)
            yield np.where(~test_mask)[0], np.where(test_mask)[0], (wt, ad)


def _train_one_l2o_fold(train_idx, test_idx, test_animals, X, y, groups, genos,
                        params, n_repeats):
    """Train one L2O fold — held-out set contains 1 WT + 1 AD animal.

    With n_repeats > 0, also returns permutation importance measured on the
    held-out animals (drop in balanced accuracy when one feature is shuffled).
    """
    mlp, transform = fit_mlp(params, X[train_idx], y[train_idx])
    X_test = transform(X[test_idx])
    preds_all = mlp.predict(X_test)
    probs_all = mlp.predict_proba(X_test)[:, 1]

    test_groups = groups[test_idx]
    records = []
    for ta in test_animals:
        mask = test_groups == ta
        anim_global = test_idx[mask]
        preds, probs = preds_all[mask], probs_all[mask]
        records.append({
            'test_idx': anim_global,
            'preds': preds,
            'probs': probs,
            'animal': ta,
            'geno': genos[anim_global][0],
            'true': int(y[anim_global][0]),
            'pred': int(np.mean(probs) >= 0.5),
            'mean_prob': float(np.mean(probs)),
            'neuron_acc': float(accuracy_score(y[anim_global], preds)),
            'n_neurons': int(len(anim_global)),
            'pair': test_animals,
        })

    importance = None
    if n_repeats > 0:
        importance = permutation_importance(
            mlp, X_test, y[test_idx], scoring='balanced_accuracy',
            n_repeats=n_repeats, random_state=42, n_jobs=1).importances_mean
    return records, importance


def run_l2o_folds(feature_df, feat_cols, age_split, region=None, cell_type=None, n_repeats=0):
    """Run L2O across all (WT, AD) pairs for a given subset.

    Returns {'records', 'mean_importance', 'std_importance', 'feature_cols'},
    or None if either genotype has fewer than 2 animals.
    """
    df_use = feature_df[~feature_df['animal'].isin(SUBSET_EXCLUDE)]
    if age_split in ('OLD', 'YOUNG'):
        df_use = df_use[df_use['age_group'] == age_split]
    if region is not None:
        df_use = df_use[df_use['brain_region'] == region]
    if cell_type is not None:
        df_use = df_use[df_use['cell_type'] == cell_type]
    df_use = df_use.reset_index(drop=True)

    n_ad = df_use[df_use['label'] == 1]['animal'].nunique()
    n_wt = df_use[df_use['label'] == 0]['animal'].nunique()
    if n_ad < 2 or n_wt < 2:
        return None

    X      = df_use[feat_cols].values.astype(float)
    y      = df_use['label'].values
    groups = df_use['animal'].values
    genos  = df_use['geno_label'].values

    with parallel_backend('loky', inner_max_num_threads=1):   # 1 BLAS thread per worker
        fold_outputs = Parallel(n_jobs=N_JOBS)(
            delayed(_train_one_l2o_fold)(tr, te, tas, X, y, groups, genos,
                                         BEST_MLP_PARAMS, n_repeats)
            for tr, te, tas in _l2o_pairs(groups, y))
    importances = [imp for _recs, imp in fold_outputs if imp is not None]
    return {
        'records': [r for recs, _imp in fold_outputs for r in recs],
        'n_folds': len(fold_outputs),
        'mean_importance': np.mean(importances, axis=0) if importances else None,
        'std_importance': np.std(importances, axis=0) if importances else None,
        'feature_cols': list(feat_cols),
    }


print('Running comprehensive feature subset analysis (LEAVE-TWO-ANIMAL-OUT).')
print('Each fold holds out 1 WT + 1 AD animal; each animal appears in '
      'multiple folds (once per opposite-genotype pairing).')
print()

subsets = [('state', name, feats, {}) for name, feats in state_feat_map.items()]
subsets += [('region', r, raw_isi_feat_cols, {'region': r}) for r in ['ACad', 'CA1', 'RSPv', 'V1']]
subsets += [('cell_type', ct, raw_isi_feat_cols, {'cell_type': ct}) for ct in CELL_TYPES]

subset_results = {}   # (age_split, category, key) -> flat list of per-animal records
mlp_runs = {}         # (age_split, category, key) -> records + fold-averaged importance
t_all = time.time()
for age_split in ['ALL', 'OLD', 'YOUNG']:
    print(f'══ Age split: {age_split} ══')
    for category, key, feats, subset in subsets:
        n_repeats = N_REPEATS if (age_split == 'ALL' and category == 'state') else 0
        t0 = time.time()
        run = run_l2o_folds(raw_isi_df, feats, age_split, n_repeats=n_repeats, **subset)
        mlp_runs[(age_split, category, key)] = run
        subset_results[(age_split, category, key)] = run['records'] if run else None
        status = (f'{len(run["records"])} animal-evals' if run
                  else 'skipped (needs >=2 animals per genotype)')
        print(f'  {category:<9} {key:<10} {status}  ({time.time() - t0:.0f}s)', flush=True)
    print()

print(f'All configurations complete in {(time.time() - t_all) / 60:.1f} min.')


## 6. Which ISI bins matter? (permutation importance)
Figure A: combined-state model, importance per bin under the mean histogram for each state. Figure B: per-state models + Spearman agreement between their importance profiles. Saves `mlp_bin_importance.csv` (used by section 10).

In [ ]:
# Held-out permutation importance from the same L2O fold models that made the predictions.
# Fold-to-fold spread is descriptive: training sets overlap.
from scipy.stats import spearmanr
df_imp = raw_isi_df[~raw_isi_df['animal'].isin(SUBSET_EXCLUDE)].reset_index(drop=True)
combined = mlp_runs[('ALL', 'state', 'All States')]
combined_mean = np.asarray(combined['mean_importance'])
combined_std = np.asarray(combined['std_importance'])
combined_feats = combined['feature_cols']
per_state = {}
for state in ['wake', 'nrem', 'rem']:
    name = {'wake': 'Wake', 'nrem': 'NREM', 'rem': 'REM'}[state]
    result = mlp_runs[('ALL', 'state', name)]
    per_state[state] = {'mean': np.asarray(result['mean_importance']),
                        'std': np.asarray(result['std_importance']),
                        'features': result['feature_cols']}
print('Importance is the held-out balanced-accuracy drop after shuffling a bin; correlated bins remain a limitation.')
STATE_COL = STATE_COLORS

# Geometric bin centers (works on log x-axis)
bin_centers = np.sqrt(BIN_EDGES[:-1] * BIN_EDGES[1:])

# ─── Mean ISI histogram per state (averaged across neurons) ─────────────────
# Pull from raw_isi_df (already normalized per neuron per state by construction)
state_mean_hist = {}
for state in ['wake', 'nrem', 'rem']:
    cols = [f'{state}_bin{b}' for b in range(N_BINS)]
    state_mean_hist[state] = df_imp[cols].mean(axis=0).values

# Unpack combined-model importance per state
def _imp_array_for_state(state, mean_vec, feats):
    """Pull the 100 importance values for one state from the combined model."""
    arr = np.zeros(N_BINS)
    for f, v in zip(feats, mean_vec):
        if f.startswith(f'{state}_bin'):
            b = int(f.split('_bin')[1])
            arr[b] = v
    return arr

combined_per_state = {s: _imp_array_for_state(s, combined_mean, combined_feats)
                     for s in ['wake', 'nrem', 'rem']}

# Per-state model importance arrays (already aligned to bin index)
per_state_imp_array = {}
for state in ['wake', 'nrem', 'rem']:
    d = per_state[state]
    arr = np.zeros(N_BINS)
    for f, v in zip(d['features'], d['mean']):
        b = int(f.split('_bin')[1])
        arr[b] = v
    per_state_imp_array[state] = arr


# Save the per-bin importance so later analyses do not have to retrain anything.
combined_sd_per_state = {s: _imp_array_for_state(s, combined_std, combined_feats)
                         for s in ['wake', 'nrem', 'rem']}
per_state_sd_array = {s: _imp_array_for_state(s, per_state[s]['std'], per_state[s]['features'])
                      for s in ['wake', 'nrem', 'rem']}

importance_rows = []
for model, mean_by_state, sd_by_state, key in [
        ('combined', combined_per_state, combined_sd_per_state, ('ALL', 'state', 'All States')),
        ('per_state', per_state_imp_array, per_state_sd_array, None)]:
    for state in ['wake', 'nrem', 'rem']:
        n_folds = mlp_runs[key or ('ALL', 'state', STATE_NAMES[state])]['n_folds']
        for b in range(N_BINS):
            importance_rows.append({
                'model': model, 'state': state, 'bin': b,
                'lo_s': BIN_EDGES[b], 'hi_s': BIN_EDGES[b + 1], 'center_s': bin_centers[b],
                'importance_mean': mean_by_state[state][b],
                'importance_sd': sd_by_state[state][b],
                'n_folds': n_folds,
                'importance_sem': sd_by_state[state][b] / np.sqrt(n_folds),
            })
bin_importance = pd.DataFrame(importance_rows)
bin_importance.to_csv('mlp_bin_importance.csv', index=False)
print(f'Saved: mlp_bin_importance.csv ({len(bin_importance)} rows)')


def _fmt_isi(t):
    if t < 0.001: return f'{t*1e6:.0f} µs'
    if t < 1:     return f'{t*1000:.0f} ms'
    return f'{t:.1f} s'


def _draw_histogram(ax, state):
    """Mean min–max-normalized ISI histogram for one state (grey reference)."""
    hist = state_mean_hist[state]
    ax.fill_between(bin_centers, 0, hist, step='mid', color=BOX_FILL, linewidth=0)
    ax.step(bin_centers, hist, where='mid', color=BOX_EDGE, lw=0.8)
    ax.set_xscale('log')
    ax.set_xlim(BIN_EDGES[0], BIN_EDGES[-1])
    ax.set_ylim(0, max(hist.max() * 1.15, 0.001))
    for ref in (0.010, 0.500):
        ax.axvline(ref, color=REF_GREY, ls=':', lw=0.8, zorder=0)


def _draw_importance(ax, state, imp_arr, n_peaks=3, min_sep=15):
    """Per-bin permutation importance; the largest peaks >= `min_sep` bins apart are labelled
    with their bin-centre ISI."""
    color = STATE_COL[state]
    ax.axhline(0, color='black', lw=0.6)
    ax.fill_between(bin_centers, 0, imp_arr, color=color, alpha=0.25, linewidth=0)
    ax.plot(bin_centers, imp_arr, color=color, lw=1.5)
    for ref in (0.010, 0.500):
        ax.axvline(ref, color=REF_GREY, ls=':', lw=0.8, zorder=0)
    peaks = []
    for ti in np.argsort(imp_arr)[::-1]:
        if len(peaks) == n_peaks:
            break
        if all(abs(ti - p) >= min_sep for p in peaks):
            peaks.append(ti)
    for ti in peaks:
        ax.plot(bin_centers[ti], imp_arr[ti], 'o', ms=3, color=color)
        ax.annotate(_fmt_isi(bin_centers[ti]),
                    xy=(bin_centers[ti], imp_arr[ti]), xytext=(0, 4), textcoords='offset points',
                    ha='center', va='bottom', fontsize=6.5, color='#333333')
    ax.set_xscale('log')
    ax.set_xlim(BIN_EDGES[0], BIN_EDGES[-1])
    ax.set_ylim(min(0, imp_arr.min() * 1.25), max(imp_arr.max() * 1.35, 0.001))


def _draw_state_grid(fig, gs, importance_by_state, title_suffix):
    """Histograms (top row) above importance curves (bottom row), one column per state."""
    axes = np.empty((2, 3), dtype=object)
    for c, state in enumerate(['wake', 'nrem', 'rem']):
        axes[0, c] = fig.add_subplot(gs[0, c])
        axes[1, c] = fig.add_subplot(gs[1, c], sharex=axes[0, c])
        _draw_histogram(axes[0, c], state)
        _draw_importance(axes[1, c], state, importance_by_state[state])
        axes[0, c].set_title(f'{STATE_NAMES[state]}{title_suffix}')
        axes[0, c].tick_params(labelbottom=False)
        axes[1, c].set_xlabel('ISI (s)')
    axes[0, 0].set_ylabel('Mean normalized\nbin count')
    axes[1, 0].set_ylabel('Permutation importance\n(Δ balanced accuracy)')
    return axes


def _draw_rank_agreement(ax, pairs, rhos, xlabel):
    """Lollipop of Spearman ρ for each pair of sleep states."""
    ypos = np.arange(len(pairs))
    ax.axvline(0, color='black', ls='--', lw=0.8)
    ax.hlines(ypos, 0, rhos, color=BOX_EDGE, lw=1.2)
    ax.scatter(rhos, ypos, s=24, color='black', zorder=3, linewidths=0)
    for y, rho in zip(ypos, rhos):
        ax.annotate(f'ρ = {rho:+.2f}', xy=(rho, y), xytext=(6 if rho >= 0 else -6, 0),
                    textcoords='offset points', ha='left' if rho >= 0 else 'right',
                    va='center', fontsize=7.5)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f'{STATE_NAMES[a]} vs {STATE_NAMES[b]}' for a, b in pairs])
    ax.set_ylim(len(pairs) - 0.4, -0.6)
    ax.set_xlim(-1, 1)
    ax.set_xticks([-1, -0.5, 0, 0.5, 1])
    ax.set_xlabel(xlabel)


# =============================================================================
# Figure A: COMBINED-model importance per state
# =============================================================================
print('Combined-state MLP. Top row: mean min–max-normalized ISI histogram per state (caf42 excluded). '
      'Bottom row: held-out permutation importance per bin (mean across L2O folds); the three largest '
      'well-separated peaks are labelled with their bin centre. Dotted lines mark 10 ms and 500 ms.')
fig = plt.figure(figsize=(7.2, 3.8), layout='constrained')
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.6])
axes = _draw_state_grid(fig, gs, combined_per_state, ' bins')
plt.savefig('fig_mlp_feature_importance.png')
plt.show()
print('Saved: fig_mlp_feature_importance.png')


# =============================================================================
# Figure B: PER-STATE models + cross-state agreement
# =============================================================================
from scipy.stats import spearmanr

# Spearman across all 100 bins, per-state vs per-state
common_bins = list(range(N_BINS))
imp_aligned = per_state_imp_array  # already aligned
pairs = [('wake', 'nrem'), ('wake', 'rem'), ('nrem', 'rem')]
rhos, pvals = [], []
for a, b in pairs:
    rho, pv = spearmanr(imp_aligned[a], imp_aligned[b])
    rhos.append(rho); pvals.append(pv)

print('Per-state MLPs. A, mean normalized ISI histogram (top) and held-out permutation importance of each '
      'state-specific model (bottom). B, Spearman ρ between the per-state importance profiles (descriptive).')
fig = plt.figure(figsize=(7.2, 5.0), layout='constrained')
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1.6, 1.1])
axes = _draw_state_grid(fig, gs, per_state_imp_array, ' model')
ax_s = fig.add_subplot(gs[2, :])
_draw_rank_agreement(ax_s, pairs, rhos, 'Spearman ρ between per-state bin-importance profiles')
panel_letters(fig, [axes[0, 0], ax_s])
plt.savefig('fig_mlp_spearman_states.png')
plt.show()
print('Saved: fig_mlp_spearman_states.png')


## 7. Result figures + summary table
`_aggregate_by_animal` / `_get_metrics` turn fold records into per-animal calls and N_Acc / N_BAcc / A_Acc. Fig 1 forest plot, Fig 2 per-animal box plots, Fig 3 score vs age, Fig 4 per-animal P(AD) ladders (4B states, 4C regions, 4D cell types).

In [ ]:
# =============================================================================
# COMPREHENSIVE FEATURE SUBSET ANALYSIS — VISUALIZATION
# =============================================================================
import matplotlib.lines as mlines
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score

# ── Shared metadata ───────────────────────────────────────────────────────────
# Equal weight per distinct recorded age, rather than selecting the first row.
_ameta = (raw_isi_df[['animal', 'age']].drop_duplicates()
          .groupby('animal')['age'].agg(['median', 'min', 'max']))
ANIMAL_AGE = _ameta['median'].to_dict()
ANIMAL_AGE_MIN = _ameta['min'].to_dict()
ANIMAL_AGE_MAX = _ameta['max'].to_dict()
ANIMAL_GENO = raw_isi_df.groupby('animal')['geno_label'].first().to_dict()

PLOT_GROUPS = [
    ('Sleep State',  [(s, 'state',     s) for s in state_feat_map.keys()]),
    ('Brain Region', [(r, 'region',    r) for r in ['ACad', 'CA1', 'RSPv', 'V1']]),
    ('Cell Type',    [(ct, 'cell_type', ct) for ct in CELL_TYPES]),
]

CONFIG_NAMES = []
CONFIG_KEYS  = []
CONFIG_GROUP = {}
for gname, configs in PLOT_GROUPS:
    for dname, category, key in configs:
        CONFIG_NAMES.append(dname)
        CONFIG_KEYS.append((category, key))
        CONFIG_GROUP[dname] = gname

# GENO_COLORS, GROUP_COLORS and the plotting helpers come from the figure-style block in setup.

AGE_SPLITS = ['ALL', 'OLD', 'YOUNG']
AGE_LABELS = {
    'ALL':   'All Animals',
    'OLD':   f'Old (>{AGE_CUTOFF}d)',
    'YOUNG': f'Young (\u2264{AGE_CUTOFF}d)',
}


def _aggregate_by_animal(fold_res):
    """Collapse L2O per-animal-eval records to one record per unique animal.

    Each animal appears in multiple L2O folds (paired with different
    opposite-genotype partners). We average P(AD) across those folds and
    derive a single per-animal call. Used by per-animal figures.
    """
    if not fold_res:
        return []
    by_animal = {}
    for f in fold_res:
        by_animal.setdefault(f['animal'], []).append(f)
    out = []
    for animal, recs in by_animal.items():
        # Weighted by neuron count (each fold's neuron_acc is on the same
        # animal's neurons, so weights are identical — but be safe).
        probs = np.array([r['mean_prob'] for r in recs])
        accs  = np.array([r['neuron_acc'] for r in recs])
        mean_prob = float(np.mean(probs))
        animal_pred = 1 if mean_prob >= 0.5 else 0
        out.append({
            'animal': animal,
            'geno':   recs[0]['geno'],
            'true':   recs[0]['true'],
            'pred':   animal_pred,
            'mean_prob': mean_prob,
            'neuron_acc': float(np.mean(accs)),
            'n_neurons':  int(recs[0]['n_neurons']),
            'n_folds':    len(recs),
        })
    return out



def _get_metrics(fold_res):
    """Compute neuron-level metrics + per-unique-animal accuracy.

    Neuron-level metrics (n_acc, n_bacc) pool predictions across all folds.
    Animal-level accuracy is computed on UNIQUE animals: aggregate each
    animal's P(AD) across the L2O folds it appeared in, then threshold.
    So a_correct/a_total reads e.g. '17/19' rather than '157/180'.
    """
    if not fold_res or len(fold_res) < 2:
        return None
    all_true  = np.concatenate([np.full(f['n_neurons'], f['true']) for f in fold_res])
    all_preds = np.concatenate([f['preds'] for f in fold_res])
    all_probs = np.concatenate([f['probs'] for f in fold_res])
    n_acc = accuracy_score(all_true, all_preds)
    try:
        n_bacc = balanced_accuracy_score(all_true, all_preds)
    except ValueError:
        n_bacc = float('nan')

    # Per-unique-animal accuracy
    agg = _aggregate_by_animal(fold_res)
    a_correct = sum(1 for r in agg if r['true'] == r['pred'])
    a_total = len(agg)

    fold_accs = np.array([f['neuron_acc'] for f in fold_res])
    # Per-unique-animal aggregated accuracy (one value per animal — for Figs 1/2)
    per_animal_accs = np.array([r['neuron_acc'] for r in agg])
    return {
        'n_acc': n_acc, 'n_bacc': n_bacc,
        'a_acc': a_correct / a_total if a_total else float('nan'),
        'a_correct': a_correct, 'a_total': a_total,
        'y_true': all_true, 'y_probs': all_probs, 'y_preds': all_preds,
        'fold_accs': fold_accs,             # one per animal-eval (L2O: ~9-10 per animal)
        'animal_accs': per_animal_accs,     # one per UNIQUE animal (aggregated)
        'fold_res': fold_res,               # raw per-eval records
        'animal_res': agg,                  # aggregated per-unique-animal records
    }




# =============================================================================
# FIGURE 1: Median and IQR of per-animal neuron accuracy, centered on 50%
# =============================================================================
print('=' * 70)
print('FIGURE 1: Forest Plot')
print('=' * 70)
print('Dot = median per-animal neuron accuracy minus 0.5; line = IQR across animals; '
      'configurations sorted best to worst within each age group.')

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), sharex=True)
x_extent = [-0.1, 0.1]   # widened below so no IQR line is clipped
for ax, age_split in zip(axes, AGE_SPLITS):
    rows = []
    for dname, (cat, key) in zip(CONFIG_NAMES, CONFIG_KEYS):
        m = _get_metrics(subset_results.get((age_split, cat, key)))
        if m is None:
            continue
        centered = m['animal_accs'] - 0.5
        rows.append({
            'name':   dname,
            'median': float(np.median(centered)),
            'q1':     float(np.percentile(centered, 25)),
            'q3':     float(np.percentile(centered, 75)),
            'group':  CONFIG_GROUP[dname],
        })
    rows.sort(key=lambda r: r['median'], reverse=True)

    for i, row in enumerate(rows):
        color = GROUP_COLORS[row['group']]
        ax.plot([row['q1'], row['q3']], [i, i], color=color, lw=1.5, solid_capstyle='butt')
        ax.scatter(row['median'], i, color=color, s=22, zorder=3, linewidths=0)
        x_extent = [min(x_extent[0], row['q1']), max(x_extent[1], row['q3'])]
    ax.axvline(0, color='black', ls='--', lw=0.8, zorder=0)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r['name'] for r in rows])
    ax.invert_yaxis()
    ax.set_xlabel('Neuron accuracy \u2212 chance')
    ax.set_title(AGE_LABELS[age_split])

axes[0].set_xlim(x_extent[0] - 0.03, x_extent[1] + 0.03)   # shared x across panels
fig.tight_layout(w_pad=2.0)
axes[-1].legend(handles=[mlines.Line2D([], [], color=c, marker='o', ms=5, lw=1.5, label=g)
                         for g, c in GROUP_COLORS.items()],
                loc='upper left', bbox_to_anchor=(1.03, 1.0))
panel_letters(fig, axes)
plt.savefig('fig1_forest.pdf')
plt.savefig('fig1_forest.png')
plt.show()
print('Saved: fig1_forest.pdf/.png')


# =============================================================================
# FIGURE 2: Boxplots with per-animal dots (sorted, genotype-colored)
# =============================================================================
print('\n' + '=' * 70)
print('FIGURE 2: Boxplot with Fold Variability')
print('=' * 70)
print('Each dot = one held-out animal (neuron accuracy averaged across its L2O pairs); '
      'box = IQR, whiskers = min/max, dashed line = chance.')

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), sharey=True)
_rng = np.random.RandomState(7)
for ax, age_split in zip(axes, AGE_SPLITS):
    rows = []
    for dname, (cat, key) in zip(CONFIG_NAMES, CONFIG_KEYS):
        m = _get_metrics(subset_results.get((age_split, cat, key)))
        if m is None:
            continue
        rows.append({
            'name':      dname,
            'median':    float(np.median(m['animal_accs'])),
            'fold_accs': m['animal_accs'],
            'fold_res':  m['animal_res'],
        })
    rows.sort(key=lambda r: r['median'], reverse=True)

    for i, row in enumerate(rows):
        ax.boxplot(
            row['fold_accs'], positions=[i], widths=0.6,
            patch_artist=True, showfliers=False, whis=(0, 100),
            medianprops=dict(color='black', linewidth=1.2),
            boxprops=dict(facecolor=BOX_FILL, edgecolor=BOX_EDGE, linewidth=0.8),
            whiskerprops=dict(color=BOX_EDGE, linewidth=0.8),
            capprops=dict(color=BOX_EDGE, linewidth=0.8),
        )
        jitter = _rng.uniform(-0.18, 0.18, len(row['fold_res']))
        ax.scatter(i + jitter, [f['neuron_acc'] for f in row['fold_res']],
                   c=[GENO_COLORS[f['geno']] for f in row['fold_res']], s=10, linewidths=0, zorder=3)

    ax.axhline(0.5, color='black', ls='--', lw=0.8, zorder=0)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels([r['name'] for r in rows], rotation=45, ha='right', rotation_mode='anchor')
    ax.set_xlim(-0.6, len(rows) - 0.4)
    ax.set_ylim(0, 1.02)
    ax.set_title(AGE_LABELS[age_split])
axes[0].set_ylabel('Per-animal neuron accuracy')

fig.tight_layout(w_pad=1.0)
axes[-1].legend(handles=genotype_handles() + [CHANCE_HANDLE],
                loc='upper left', bbox_to_anchor=(1.03, 1.0))
panel_letters(fig, axes)
plt.savefig('fig2_boxplot_folds.pdf')
plt.savefig('fig2_boxplot_folds.png')
plt.show()
print('Saved: fig2_boxplot_folds.pdf/.png')


# =============================================================================
# FIGURE 3: Descriptive age association of uncalibrated model scores
# =============================================================================
AGE_LEGEND = [
    mlines.Line2D([], [], color=GENO_COLORS['AD'], marker='^', ls='', ms=5, label='AD (APP/PS1)'),
    mlines.Line2D([], [], color=GENO_COLORS['WT'], marker='o', ls='', ms=5, label='WT'),
]
MISCLASSIFIED_HANDLES = [
    mlines.Line2D([], [], color='white', marker='o', markeredgecolor='black', ls='', ms=5,
                  label='Misclassified'),
    mlines.Line2D([], [], color='black', ls='--', lw=0.8, label='Decision threshold'),
]


def _draw_age_scores(ax, ages, probs, genos, names, colors):
    """Per-animal mean AD score vs age with a descriptive linear fit per genotype. Returns slopes."""
    ax.axhline(0.5, color='black', ls='--', lw=0.8, zorder=1)
    ax.axvline(AGE_CUTOFF, color=REF_GREY, ls=':', lw=0.8, zorder=1)
    ax.text(AGE_CUTOFF, 1.0, f' {AGE_CUTOFF} d', color=REF_GREY, fontsize=7, ha='left', va='top')

    slopes = {}
    age_grid = np.linspace(ages.min() - 20, ages.max() + 20, 200)
    for geno in ['AD', 'WT']:
        mask = genos == geno
        if mask.sum() < 2:
            continue
        slope, intercept = np.polyfit(ages[mask], probs[mask], 1)
        slopes[geno] = slope
        ax.plot(age_grid, slope * age_grid + intercept, color=GENO_COLORS[geno], lw=1.5, zorder=2)

    for age, prob, geno, name, color in zip(ages, probs, genos, names, colors):
        correct = (geno == 'AD' and prob >= 0.5) or (geno == 'WT' and prob < 0.5)
        ax.errorbar(age, prob, xerr=[[age - ANIMAL_AGE_MIN[name]], [ANIMAL_AGE_MAX[name] - age]],
                    fmt='none', ecolor='#BDBDBD', elinewidth=0.8, zorder=2)
        ax.scatter(age, prob, marker='^' if geno == 'AD' else 'o', s=32, color=color,
                   edgecolors='none' if correct else 'black', linewidths=1.0, zorder=4)
        ax.annotate(name, xy=(age, prob), xytext=(4, 0), textcoords='offset points',
                    fontsize=6, color='#6E6E6E', va='center')

    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('Age (days; median of recordings, bars = range)')
    ax.set_ylabel('Mean AD score (uncalibrated)')
    if 'AD' in slopes and 'WT' in slopes:
        stat_box(ax, f'Slope per 100 d\nAD {slopes["AD"] * 100:+.3f}\nWT {slopes["WT"] * 100:+.3f}',
                 x=1.03, y=0.0, ha='left', va='bottom')
    return slopes


print('\n' + '=' * 70)
print('FIGURE 3: Age vs Model Confidence')
print('=' * 70)
print('Mean uncalibrated AD score per animal (held-out, averaged across L2O pairs) against median '
      'recorded age; horizontal bars show the recorded age range. Lines are descriptive least-squares '
      'fits per genotype (no sex adjustment). Black outline = misclassified.')

# Use ALL age split + All States config
fold_res_all = subset_results.get(('ALL', 'state', 'All States'))
m_all = _get_metrics(fold_res_all)

# Aggregate per animal — each animal appears in multiple L2O folds
fold_res_agg = _aggregate_by_animal(fold_res_all)
ages   = np.array([ANIMAL_AGE[f['animal']] for f in fold_res_agg])
probs  = np.array([f['mean_prob']          for f in fold_res_agg])
genos  = np.array([f['geno']               for f in fold_res_agg])
names  = [f['animal'] for f in fold_res_agg]

fig, ax = plt.subplots(figsize=(4.6, 3.4))
slopes = _draw_age_scores(ax, ages, probs, genos, names, colors=[GENO_COLORS[g] for g in genos])
ax.set_title('All animals, all-state features')
ax.legend(handles=AGE_LEGEND + MISCLASSIFIED_HANDLES, loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.savefig('fig3_age_confidence.pdf')
plt.savefig('fig3_age_confidence.png')
plt.show()
print('Saved: fig3_age_confidence.pdf/.png')


# =============================================================================
# FIGURE 4: Per-animal AD score ladders
# =============================================================================
LADDER_HANDLES = [Patch(facecolor=GENO_COLORS['AD'], label='AD (APP/PS1)'),
                  Patch(facecolor=GENO_COLORS['WT'], label='WT'),
                  Patch(facecolor='white', edgecolor='black', label='Misclassified')]


def _draw_score_ladder(ax, fold_res, title):
    """Per-animal mean AD score, sorted; misclassified animals are outlined and marked with ×."""
    ax.set_title(title)
    ax.set_xlim(0, 1)
    if not fold_res:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, color=REF_GREY)
        return False
    fold_sorted = sorted(_aggregate_by_animal(fold_res), key=lambda f: f['mean_prob'])
    probs = [f['mean_prob'] for f in fold_sorted]
    wrong = [f['true'] != f['pred'] for f in fold_sorted]
    ypos = np.arange(len(fold_sorted))
    ax.barh(ypos, probs, height=0.7, color=[GENO_COLORS[f['geno']] for f in fold_sorted],
            edgecolor=['black' if w else 'none' for w in wrong],
            linewidth=[1.0 if w else 0 for w in wrong])
    for y, p, w in zip(ypos, probs, wrong):
        if w:
            ax.text(min(p + 0.02, 0.95), y, '\u00d7', ha='left', va='center', fontsize=8)
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{f['animal']} ({ANIMAL_AGE[f['animal']]:.0f} d)" for f in fold_sorted],
                       fontsize=7)
    ax.axvline(0.5, color='black', ls='--', lw=0.8)
    ax.set_xlabel('Mean AD score (uncalibrated)')
    m = _get_metrics(fold_res)
    if m:
        ax.set_title(f'{title}\n{m["a_correct"]}/{m["a_total"]} animals, neuron acc. {m["n_acc"]:.1%}',
                     fontsize=8)
    return True


print('\n' + '=' * 70)
print('FIGURE 4: Per-Animal P(AD) \u2014 All States')
print('=' * 70)
print('Mean uncalibrated AD score per held-out animal (averaged across L2O pairs), all-state '
      'histogram features; outline and \u00d7 = misclassified.')

fig, axes = plt.subplots(1, 3, figsize=(7.5, 3.8))
for ax, age_split in zip(axes, AGE_SPLITS):
    _draw_score_ladder(ax, subset_results.get((age_split, 'state', 'All States')), AGE_LABELS[age_split])
fig.tight_layout(w_pad=1.0)
axes[-1].legend(handles=LADDER_HANDLES, loc='upper left', bbox_to_anchor=(1.03, 1.0))
panel_letters(fig, axes)
plt.savefig('fig4_animal_predictions.pdf')
plt.savefig('fig4_animal_predictions.png')
plt.show()
print('Saved: fig4_animal_predictions.pdf/.png')


# =============================================================================
# FIGURE 4B/4C/4D: Per-animal AD score — ONE FIGURE PER CONDITION
# =============================================================================
# Each condition (each sleep state / brain region / cell type) gets its own
# standalone figure with 3 age-split panels (ALL / OLD / YOUNG).

def _plot_per_animal_for_condition(category, key, label, group_label, save_prefix):
    """Render one standalone figure for a single condition across age splits."""
    print('\n' + '=' * 70)
    print(f'{group_label}: Per-Animal P(AD) — {label}')
    print('=' * 70)

    fig, axes = plt.subplots(1, len(AGE_SPLITS), figsize=(7.5, 3.8))
    any_data = False
    for ax, age_split in zip(axes, AGE_SPLITS):
        drawn = _draw_score_ladder(ax, subset_results.get((age_split, category, key)),
                                   f'{label} \u00b7 {AGE_LABELS[age_split]}')
        any_data = any_data or drawn
    fig.tight_layout(w_pad=1.0)
    axes[-1].legend(handles=LADDER_HANDLES, loc='upper left', bbox_to_anchor=(1.03, 1.0))
    panel_letters(fig, axes)
    plt.savefig(f'{save_prefix}.pdf')
    plt.savefig(f'{save_prefix}.png')
    plt.show()
    print(f'Saved: {save_prefix}.pdf/.png')
    return any_data


# Sleep states — one figure each
for state_name in ['Wake', 'NREM', 'REM']:
    _plot_per_animal_for_condition(
        category='state', key=state_name, label=state_name,
        group_label='FIGURE 4B', save_prefix=f'fig4b_state_{state_name.lower()}'
    )

# Brain regions — one figure each
for region_name in ['ACad', 'CA1', 'RSPv', 'V1']:
    _plot_per_animal_for_condition(
        category='region', key=region_name, label=region_name,
        group_label='FIGURE 4C', save_prefix=f'fig4c_region_{region_name.lower()}'
    )

# Cell types — one figure each
for celltype_name in ['FS', 'RSU']:
    _plot_per_animal_for_condition(
        category='cell_type', key=celltype_name, label=celltype_name,
        group_label='FIGURE 4D', save_prefix=f'fig4d_celltype_{celltype_name.lower()}'
    )


# =============================================================================
# Summary table
# =============================================================================
print('\n' + '=' * 70)
print('SUMMARY TABLE')
print('=' * 70)
for age_split in AGE_SPLITS:
    print(f'\n  {AGE_LABELS[age_split]}')
    print(f'  {"Config":<15} {"N_Acc":>8} {"N_BAcc":>8} {"A_Acc":>8}  {"Animals":>8}')
    print(f'  {"-"*55}')
    for gname, configs in PLOT_GROUPS:
        for dname, category, key in configs:
            m = _get_metrics(subset_results.get((age_split, category, key)))
            if not m:
                print(f'  {dname:<15} {"n/a":>8} {"n/a":>8} {"n/a":>8}')
                continue
            print(f'  {dname:<15} {m["n_acc"]:>7.1%} {m["n_bacc"]:>8.1%} {m["a_acc"]:>7.1%}  ({m["a_correct"]}/{m["a_total"]})')
    print()


## 8. Score vs age, shaded by sex
Same data as Fig 3.

In [ ]:
# =============================================================================
# FIGURE 3B: Age vs Model Confidence + Sex Shades
# =============================================================================
print('\n' + '=' * 70)
print('FIGURE 3B: Age vs Model Confidence + Sex')
print('=' * 70)

_sex_meta = (df[['animal', 'sex']]
             .drop_duplicates(subset=['animal'])
             .sort_values('animal')
             .reset_index(drop=True))
ANIMAL_SEX = {r['animal']: r['sex'] for _, r in _sex_meta.iterrows()}

FALLBACK_COLOR = '#888888'

# Use ALL age split + All States config (same data as Figure 3)
fold_res_all = subset_results.get(('ALL', 'state', 'All States'))

# Aggregate per animal — each animal appears in multiple L2O folds
fold_res_agg = _aggregate_by_animal(fold_res_all)
ages   = np.array([ANIMAL_AGE[f['animal']] for f in fold_res_agg])
probs  = np.array([f['mean_prob']          for f in fold_res_agg])
genos  = np.array([f['geno']               for f in fold_res_agg])
sexes  = np.array([ANIMAL_SEX.get(f['animal'], '?') for f in fold_res_agg])
names  = [f['animal'] for f in fold_res_agg]

print('Same data as Figure 3; shade marks sex within each genotype (the fits do not adjust for sex).')
fig, ax = plt.subplots(figsize=(4.6, 3.4))
slopes = _draw_age_scores(ax, ages, probs, genos, names,
                          colors=[SEX_COLORS.get((g, s), FALLBACK_COLOR) for g, s in zip(genos, sexes)])
ax.set_title('All animals, all-state features, by sex')
sex_handles = [
    mlines.Line2D([], [], color=SEX_COLORS[(g, s)], marker='^' if g == 'AD' else 'o', ls='', ms=5,
                  label=f'{g} {"female" if s == "F" else "male"}')
    for g, s in [('AD', 'F'), ('AD', 'M'), ('WT', 'F'), ('WT', 'M')]
]
ax.legend(handles=sex_handles + MISCLASSIFIED_HANDLES, loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.savefig('fig3_age_confidence_sex.pdf')
plt.savefig('fig3_age_confidence_sex.png')
plt.show()
print('Saved: fig3_age_confidence_sex.pdf/.png')


## 9. QC deep-dive: `caf73`
Repeats the checks that flagged `caf104` (negative / zero / sub-ms / > 60 s / > 100 s ISIs, state balance, how many neurons have ≥ 10 ISIs per state) for `caf73` vs its same-age, same-genotype peers, plus its mean histogram vs peers.

In [ ]:
# =============================================================================
# DATA QUALITY INVESTIGATION — caf73
# =============================================================================
# Check whether caf73 has raw ISI data issues that could explain poor
# classification, mirroring the checks that flagged caf104 for exclusion.
TARGET = 'caf73'

if TARGET not in df['animal'].unique():
    print(f'{TARGET} not found in dataset.')
else:
    target_df = df[df['animal'] == TARGET]
    target_row = target_df.iloc[0]
    true_geno = target_row['geno_label']
    true_age  = target_row['age']
    true_group = target_row['age_group']

    print('=' * 70)
    print(f'BASIC PROFILE: {TARGET}')
    print('=' * 70)
    print(f'  Genotype:      {target_row["geno"]} ({true_geno})')
    print(f'  Age:           {int(true_age)} days  ({true_group})')
    print(f'  N neurons:     {len(target_df):,}')
    print(f'  Brain regions: {dict(target_df["brain_region"].value_counts())}')
    print(f'  Cell types:    {dict(target_df["cell_type"].value_counts())}')

    # Peers = same genotype + same age group, excluding caf73
    peer_df = df[(df['age_group'] == true_group) &
                 (df['geno_label'] == true_geno) &
                 (df['animal'] != TARGET)]
    print(f'  Peers ({true_group} {true_geno}): {peer_df["animal"].nunique()} animals, {len(peer_df):,} neurons')

    # ── Per-state ISI quality summary ─────────────────────────────────────────
    def _isi_quality_stats(rows, col):
        """Collect raw ISI stats from a slice of the dataframe."""
        all_isis = []
        n_empty_neurons = 0
        neuron_epoch_counts = []
        for _, r in rows.iterrows():
            neuron_total = 0
            for arr in r[col]:
                if isinstance(arr, np.ndarray) and len(arr) > 0:
                    all_isis.append(arr)
                    neuron_total += len(arr)
            if neuron_total == 0:
                n_empty_neurons += 1
            else:
                neuron_epoch_counts.append(neuron_total)
        if len(all_isis) == 0:
            return None
        all_isis = np.concatenate(all_isis)
        total = len(all_isis)
        return {
            'n_isis':         total,
            'n_zero':         int(np.sum(all_isis == 0)),
            'n_negative':     int(np.sum(all_isis < 0)),
            'n_sub_ms':       int(np.sum((all_isis > 0) & (all_isis < 0.001))),
            'n_over_60s':     int(np.sum(all_isis > 60)),
            'n_over_100s':    int(np.sum(all_isis > 100)),
            'n_inf':          int(np.sum(np.isinf(all_isis))),
            'n_nan':          int(np.sum(np.isnan(all_isis))),
            'median_isi':     float(np.median(all_isis)),
            'p90_isi':        float(np.percentile(all_isis, 90)),
            'p99_isi':        float(np.percentile(all_isis, 99)),
            'max_isi':        float(all_isis.max()),
            'n_empty_neurons': n_empty_neurons,
            'neurons_with_data': len(neuron_epoch_counts),
            'median_isis_per_neuron': float(np.median(neuron_epoch_counts)) if neuron_epoch_counts else 0,
        }

    print()
    print('=' * 70)
    print('ISI QUALITY PER STATE')
    print('=' * 70)
    for state in ['wake', 'nrem', 'rem']:
        col = f'{state}_isi'
        t_stats = _isi_quality_stats(target_df, col)
        p_stats = _isi_quality_stats(peer_df, col)
        print(f'\n  {state.upper()}')
        if t_stats is None:
            print(f'    {TARGET}: NO DATA (suspicious!)')
        else:
            print(f'    {TARGET:<10} n={t_stats["n_isis"]:>10,}  median={t_stats["median_isi"]*1000:>6.1f}ms  '
                  f'p90={t_stats["p90_isi"]:>6.2f}s  p99={t_stats["p99_isi"]:>6.2f}s  max={t_stats["max_isi"]:>7.1f}s')
            print(f'               zeros={t_stats["n_zero"]:<6} sub-1ms={t_stats["n_sub_ms"]:<6}  '
                  f'>60s={t_stats["n_over_60s"]:<6} >100s={t_stats["n_over_100s"]:<6} '
                  f'inf={t_stats["n_inf"]} nan={t_stats["n_nan"]}')
            print(f'               empty_neurons={t_stats["n_empty_neurons"]} / {len(target_df)}  '
                  f'median_isis/neuron={t_stats["median_isis_per_neuron"]:.0f}')
            # Flag if any fraction > 1%
            frac_sub = t_stats['n_sub_ms'] / max(t_stats['n_isis'], 1)
            frac_big = t_stats['n_over_60s'] / max(t_stats['n_isis'], 1)
            frac_huge = t_stats['n_over_100s'] / max(t_stats['n_isis'], 1)
            flags = []
            if frac_sub > 0.01:  flags.append(f'sub-1ms {frac_sub:.1%}')
            if frac_big > 0.01:  flags.append(f'>60s {frac_big:.1%}')
            if frac_huge > 0.005: flags.append(f'>100s {frac_huge:.1%}')
            if t_stats['n_empty_neurons'] / max(len(target_df), 1) > 0.1:
                flags.append(f'{t_stats["n_empty_neurons"]} empty neurons')
            if flags:
                print(f'               \u26A0  FLAGS: {", ".join(flags)}')
        if p_stats is not None:
            print(f'    {"peers":<10} n={p_stats["n_isis"]:>10,}  median={p_stats["median_isi"]*1000:>6.1f}ms  '
                  f'p90={p_stats["p90_isi"]:>6.2f}s  p99={p_stats["p99_isi"]:>6.2f}s  max={p_stats["max_isi"]:>7.1f}s')
            print(f'               zeros={p_stats["n_zero"]:<6} sub-1ms={p_stats["n_sub_ms"]:<6}  '
                  f'>60s={p_stats["n_over_60s"]:<6} >100s={p_stats["n_over_100s"]:<6}')
            print(f'               median_isis/neuron={p_stats["median_isis_per_neuron"]:.0f}')

    # ── State balance: does caf73 have comparable amounts of each state? ─────
    print()
    print('=' * 70)
    print('STATE BALANCE (total ISIs per state, excluding hardware-floor and >60s)')
    print('=' * 70)
    for src_name, src in [(TARGET, target_df), (f'peers ({true_group} {true_geno})', peer_df)]:
        counts = {}
        for state in ['wake', 'nrem', 'rem']:
            col = f'{state}_isi'
            total = 0
            for _, r in src.iterrows():
                for arr in r[col]:
                    if isinstance(arr, np.ndarray) and len(arr) > 0:
                        total += int(np.sum((arr > 0.001) & (arr <= 60)))
            counts[state] = total
        tot = sum(counts.values()) or 1
        print(f'  {src_name}')
        for state in ['wake', 'nrem', 'rem']:
            pct = counts[state] / tot * 100
            print(f'    {state:<6} {counts[state]:>12,}  ({pct:5.1f}%)')

    # ── Check how many caf73 neurons pass the "< 10 ISIs" histogram threshold ─
    print()
    print('=' * 70)
    print('HISTOGRAM-READY NEURONS (>= 10 ISIs in state → contributes a real histogram)')
    print('=' * 70)
    def _count_ready(rows):
        ready = {'wake': 0, 'nrem': 0, 'rem': 0}
        total = len(rows)
        for _, r in rows.iterrows():
            for state in ['wake', 'nrem', 'rem']:
                col = f'{state}_isi'
                n = 0
                for arr in r[col]:
                    if isinstance(arr, np.ndarray) and len(arr) > 0:
                        valid = arr[(arr > 0) & (arr <= 60)]
                        n += len(valid)
                if n >= 10:
                    ready[state] += 1
        return ready, total
    t_ready, t_total = _count_ready(target_df)
    p_ready, p_total = _count_ready(peer_df)
    for state in ['wake', 'nrem', 'rem']:
        t_pct = 100 * t_ready[state] / max(t_total, 1)
        p_pct = 100 * p_ready[state] / max(p_total, 1)
        print(f'  {state.upper():<6}  {TARGET}: {t_ready[state]:>5}/{t_total} ({t_pct:5.1f}%)   '
              f'peers: {p_ready[state]:>5}/{p_total} ({p_pct:5.1f}%)')

    # ── Visualize: caf73 average ISI histogram vs peers ──────────────────────
    target_isi_rows = raw_isi_df[raw_isi_df['animal'] == TARGET]
    peer_animals = peer_df['animal'].unique()
    peer_isi_rows = raw_isi_df[raw_isi_df['animal'].isin(peer_animals)]

    print(f'{TARGET} mean normalized ISI histogram (± SD across neurons) against same-age, '
          f'same-genotype peers.')
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6), sharey=True)
    bin_centers = np.sqrt(BIN_EDGES[:-1] * BIN_EDGES[1:])  # geometric center
    target_color = GENO_COLORS.get(true_geno, 'black')

    for ax, state in zip(axes, ['wake', 'nrem', 'rem']):
        bin_cols = [f'{state}_bin{b}' for b in range(N_BINS)]
        if len(target_isi_rows) and len(peer_isi_rows):
            t_mean = target_isi_rows[bin_cols].mean().values
            t_std  = target_isi_rows[bin_cols].std().values
            p_mean = peer_isi_rows[bin_cols].mean().values
            p_std  = peer_isi_rows[bin_cols].std().values

            ax.fill_between(bin_centers, t_mean - t_std, t_mean + t_std,
                            color=target_color, alpha=0.2, linewidth=0)
            ax.plot(bin_centers, t_mean, color=target_color, lw=1.5, label=TARGET)
            ax.fill_between(bin_centers, p_mean - p_std, p_mean + p_std,
                            color='#BDBDBD', alpha=0.35, linewidth=0)
            ax.plot(bin_centers, p_mean, color='black', lw=1.2, ls='--',
                    label=f'Peers ({true_group} {true_geno}, n={len(peer_animals)})')

        ax.set_xscale('log')
        ax.set_xlabel('ISI (s)')
        ax.set_title(STATE_NAMES[state])
    axes[0].set_ylabel('Normalized bin count\n(mean ± SD)')

    fig.tight_layout()
    axes[-1].legend(loc='upper left', bbox_to_anchor=(1.03, 1.0))
    panel_letters(fig, axes)
    plt.savefig(f'investigate_{TARGET}_histogram.pdf')
    plt.savefig(f'investigate_{TARGET}_histogram.png')
    plt.show()


## 10. Restricted-bin MLPs
Picks, per state, the contiguous run of bins whose importance is > 2 SEM above zero with the largest summed importance, retrains L2O on only those bins, and compares with equal-sized least-important and random-bin controls.

In [ ]:
# =============================================================================
# RESTRICTED-BIN MLPs — does one ISI window carry the genotype signal?
# =============================================================================
# Windows come from a fixed rule applied to the saved combined-model importance:
# contiguous runs of bins whose mean importance is more than SEM_THRESHOLD SEM
# above zero. The widest-summing run per state is the "most important window".
# Controls keep the bin COUNT identical, so a drop cannot just mean "fewer features".

SEM_THRESHOLD = 2.0
MIN_WINDOW_BINS = 2
N_RANDOM_CONTROLS = 3


def _significant_windows(mean_imp, sem_imp, threshold=SEM_THRESHOLD, min_width=MIN_WINDOW_BINS):
    """Contiguous bin runs whose mean importance sits `threshold` SEM above zero."""
    keep = (mean_imp - threshold * sem_imp) > 0
    windows, start = [], None
    for b, flag in enumerate(list(keep) + [False]):
        if flag and start is None:
            start = b
        elif not flag and start is not None:
            if b - start >= min_width:
                windows.append((start, b - 1))
            start = None
    return windows


imp = bin_importance[bin_importance['model'] == 'combined']
best_window, window_cols = {}, []
print(f'Windows with mean importance > {SEM_THRESHOLD} SEM above zero (combined-state model):')
for state in ['wake', 'nrem', 'rem']:
    s = imp[imp['state'] == state].sort_values('bin')
    windows = _significant_windows(s['importance_mean'].values, s['importance_sem'].values)
    if not windows:
        print(f'  {STATE_NAMES[state]:<5} none')
        continue
    scored = [(s['importance_mean'].values[a:b + 1].sum(), a, b) for a, b in windows]
    total, a, b = max(scored)
    best_window[state] = (a, b)
    window_cols += [f'{state}_bin{i}' for i in range(a, b + 1)]
    others = ', '.join(f'{_fmt_isi(BIN_EDGES[x])}–{_fmt_isi(BIN_EDGES[y + 1])}'
                       for _t, x, y in sorted(scored, reverse=True)[1:4])
    print(f'  {STATE_NAMES[state]:<5} {len(windows)} window(s); widest-summing = bins {a}-{b} '
          f'({_fmt_isi(BIN_EDGES[a])}–{_fmt_isi(BIN_EDGES[b + 1])}, {b - a + 1} bins, Σ={total:.4f})'
          + (f'; next: {others}' if others else ''))

if not window_cols:
    raise RuntimeError('No bin passed the rule — lower SEM_THRESHOLD or switch to a top-N rule.')

n_per_state = {s: (b - a + 1) for s, (a, b) in best_window.items()}
print(f'\nSelected {len(window_cols)} of {len(raw_isi_feat_cols)} bins: '
      + ', '.join(f'{STATE_NAMES[s]} {n}' for s, n in n_per_state.items()))
print('For comparison, Tolossa et al. (eLife 2025) found <~150 ms mattered most for midbrain '
      'and 200–600 ms for visual cortex, using 100 uniform bins over 0–3 s.')

# ---- control feature sets of the same size ---------------------------------
least_cols = []
for state, n in n_per_state.items():
    s = imp[imp['state'] == state].sort_values('importance_mean')
    least_cols += [f'{state}_bin{b}' for b in s['bin'].values[:n]]

random_sets = []
for draw in range(N_RANDOM_CONTROLS):
    rng = np.random.RandomState(100 + draw)
    cols = []
    for state, n in n_per_state.items():
        cols += [f'{state}_bin{b}' for b in rng.choice(N_BINS, size=n, replace=False)]
    random_sets.append(cols)

# ---- run L2O for each feature set -------------------------------------------
variants = [('All 300 bins', raw_isi_feat_cols),
            ('Important window', window_cols),
            ('Least important', least_cols)]
variants += [(f'Random bins {i + 1}', cols) for i, cols in enumerate(random_sets)]

restricted_runs = {}
print()
for name, cols in variants:
    if name == 'All 300 bins' and mlp_runs.get(('ALL', 'state', 'All States')):
        restricted_runs[name] = mlp_runs[('ALL', 'state', 'All States')]   # reuse, identical setup
    else:
        restricted_runs[name] = run_l2o_folds(raw_isi_df, cols, 'ALL')
    m = _get_metrics(restricted_runs[name]['records'])
    print(f'  {name:<18} {len(cols):>3} bins  neuron BAcc {m["n_bacc"]:.1%}  '
          f'animals {m["a_correct"]}/{m["a_total"]}', flush=True)

# ---- figure ------------------------------------------------------------------
print('\nEach bar = one feature set; dots = individual held-out animals. Controls use the same '
      'number of bins as the important window, so differences are not a feature-count effect.')
names = [n for n, _ in variants]
metrics = {n: _get_metrics(restricted_runs[n]['records']) for n in names}

fig, ax = plt.subplots(figsize=(5.8, 3.2))
xpos = np.arange(len(names))
ax.bar(xpos, [metrics[n]['n_bacc'] for n in names], width=0.65,
       color=BOX_FILL, edgecolor=BOX_EDGE, linewidth=0.8)
_rng = np.random.RandomState(7)
for x, n in zip(xpos, names):
    accs = [r['neuron_acc'] for r in metrics[n]['animal_res']]
    genos = [r['geno'] for r in metrics[n]['animal_res']]
    ax.scatter(x + _rng.uniform(-0.18, 0.18, len(accs)), accs,
               c=[GENO_COLORS[g] for g in genos], s=10, linewidths=0, zorder=3)
    ax.text(x, 1.02, f'{metrics[n]["a_correct"]}/{metrics[n]["a_total"]}',
            ha='center', va='bottom', fontsize=7)
ax.axhline(0.5, color='black', ls='--', lw=0.8, zorder=0)
ax.set_xticks(xpos)
ax.set_xticklabels([f'{n} ({len(c)})' for n, c in variants], fontsize=7,
                   rotation=20, ha='right', rotation_mode='anchor')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Neuron balanced accuracy')
ax.set_xlabel('Feature set (number of bins)')
ax.set_title('All animals, L2O: restricted ISI bins vs equal-sized controls')
ax.legend(handles=genotype_handles() + [CHANCE_HANDLE], loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.savefig('fig_mlp_restricted_bins.png')
plt.show()
print('Saved: fig_mlp_restricted_bins.png  (animals correct printed above each bar)')


## 11. Exploratory PCA / LDA
Unsupervised view of the histograms (PCA, k-means ARI vs genotype) and a best-case supervised projection (LDA fit on all animals — not a held-out result).

In [ ]:
# =============================================================================
# PCA / LDA of the ISI histograms — exploratory, following Tolossa et al. (eLife 2025)
# =============================================================================
# They ran PCA and LDA on the same kind of features, plotted 2D scatters colored by
# the ground-truth label, capped display values at the 1st/99th percentile, and used
# k-means (k = number of classes) plus the adjusted Rand index (ARI) to ask whether
# the labels actually emerge as clusters. ARI ~0 means no label structure, 1 means perfect.
# LDA here is fit on ALL animals, so it is a best case and NOT a held-out result.

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler as _Scaler

work = raw_isi_df[~raw_isi_df['animal'].isin(SUBSET_EXCLUDE)].reset_index(drop=True)
X_dr = _Scaler().fit_transform(work[raw_isi_feat_cols].values.astype(float))
geno = work['geno_label'].values
group4 = np.array([f'{g} {a.lower()}' for g, a in zip(geno, work['age_group'].values)])


def _cap(arr):
    """Cap to the 1st/99th percentile for display only (as in Tolossa et al.)."""
    lo, hi = np.percentile(arr, [1, 99], axis=0)
    return np.clip(arr, lo, hi)


pca = PCA(n_components=2).fit(X_dr)
pcs = _cap(pca.transform(X_dr))
lda4 = LinearDiscriminantAnalysis(n_components=2).fit(X_dr, group4)
lds = _cap(lda4.transform(X_dr))
lda2 = LinearDiscriminantAnalysis(n_components=1).fit(X_dr, geno)
ld_geno = _cap(lda2.transform(X_dr))[:, 0]

ari_pca = adjusted_rand_score(geno, KMeans(2, n_init=10, random_state=42).fit_predict(pcs))
ari_lda = adjusted_rand_score(geno, KMeans(2, n_init=10, random_state=42).fit_predict(ld_geno.reshape(-1, 1)))

animal_mean = work.groupby('animal')[raw_isi_feat_cols].mean()
animal_geno = work.groupby('animal')['geno_label'].first().reindex(animal_mean.index).values
pcs_animal = PCA(n_components=2).fit_transform(_Scaler().fit_transform(animal_mean.values))

print(f'PCA on {X_dr.shape[0]} neurons x {X_dr.shape[1]} bins: '
      f'PC1 {pca.explained_variance_ratio_[0]:.1%}, PC2 {pca.explained_variance_ratio_[1]:.1%} of variance.')
print(f'k-means (k=2) vs genotype — ARI in PCA space {ari_pca:+.3f}, in LDA space {ari_lda:+.3f} '
      '(0 = labels are not clusters).')
print('A, neurons in PCA space. B, animal-mean histograms in PCA space. C, LDA fit to genotype x age '
      'group. D, genotype LDA axis. C and D are fit on all animals, so they show the best case, not '
      'held-out separation.')

fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.6))
for g in ['WT', 'AD']:
    m = geno == g
    axes[0, 0].scatter(pcs[m, 0], pcs[m, 1], s=4, linewidths=0, alpha=0.5,
                       color=GENO_COLORS[g], label=g)
axes[0, 0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})')
axes[0, 0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})')
axes[0, 0].set_title('Neurons (PCA)')
stat_box(axes[0, 0], f'ARI {ari_pca:+.2f}')

for g in ['WT', 'AD']:
    m = animal_geno == g
    axes[0, 1].scatter(pcs_animal[m, 0], pcs_animal[m, 1], s=34, linewidths=0,
                       color=GENO_COLORS[g], marker='^' if g == 'AD' else 'o')
for name, (x, y) in zip(animal_mean.index, pcs_animal):
    axes[0, 1].annotate(name, xy=(x, y), xytext=(4, 0), textcoords='offset points',
                        fontsize=6, color='#6E6E6E', va='center')
axes[0, 1].set_xlabel('PC1')
axes[0, 1].set_ylabel('PC2')
axes[0, 1].set_title('Animal-mean histograms (PCA)')

GROUP4_MARKERS = {'WT young': 'o', 'WT old': 's', 'AD young': '^', 'AD old': 'D'}
for label, marker in GROUP4_MARKERS.items():
    m = group4 == label
    if not m.any():
        continue
    axes[1, 0].scatter(lds[m, 0], lds[m, 1], s=4, linewidths=0, alpha=0.5, marker=marker,
                       color=GENO_COLORS[label.split()[0]], label=label)
axes[1, 0].set_xlabel('LD1')
axes[1, 0].set_ylabel('LD2')
axes[1, 0].set_title('Neurons (LDA, genotype x age group)')
axes[1, 0].legend(loc='upper left', bbox_to_anchor=(0, -0.28), ncol=2, fontsize=7)

bins = np.linspace(ld_geno.min(), ld_geno.max(), 60)
for g in ['WT', 'AD']:
    axes[1, 1].hist(ld_geno[geno == g], bins=bins, density=True, histtype='stepfilled',
                    color=GENO_COLORS[g], alpha=0.45, label=g)
    axes[1, 1].hist(ld_geno[geno == g], bins=bins, density=True, histtype='step',
                    color=GENO_COLORS[g], lw=1.2)
axes[1, 1].set_xlabel('Genotype LD axis (fit on all animals)')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Genotype LDA projection')
stat_box(axes[1, 1], f'ARI {ari_lda:+.2f}', x=0.03, ha='left')

axes[0, 0].legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), markerscale=2)
fig.tight_layout(w_pad=2.0, h_pad=2.0)
panel_letters(fig, axes)
plt.savefig('fig_mlp_pca_lda.pdf')
plt.savefig('fig_mlp_pca_lda.png')
plt.show()
print('Saved: fig_mlp_pca_lda.pdf/.png')
